# 🎛️ W11: 제조업 종합 AI 대시보드
**hexa-1 — Week 11** | ⏱️ 90분 | 🎯 W1~W10 결과를 한 화면에 통합

> W5 KPI + W6 OEE + W9 이상감지를 하나의 종합 대시보드로 통합합니다.

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q pandas matplotlib
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
nanum=next((f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name),None)
if nanum: plt.rcParams['font.family']=fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus']=False
print('✅ 준비 완료')


## Step 1: 공장 설정

In [ ]:
FACTORY = {'name':'✏️ [교육 기업명]', 'target_oee':85.0, 'target_defect':2.0}
import requests, io
url='https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
try:
    df=pd.read_csv(url,encoding='utf-8-sig'); print(f'✅ 데이터 {len(df)}행')
except:
    from google.colab import files; up=files.upload()
    df=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')
date_col=next((c for c in df.columns if '날짜' in c),df.columns[0])
total_col=next((c for c in df.columns if '생산량' in c),None)
defect_col=next((c for c in df.columns if '불량' in c and '수' in c),None)
df[date_col]=pd.to_datetime(df[date_col])
df['가동률']=df['가동시간']/df['계획시간']*100
df['불량률']=df[defect_col]/df[total_col]*100
df['OEE']=df['가동률']*(df[total_col]-df[defect_col])/df[total_col]*100*0.95
mean=df['불량률'].mean(); std=df['불량률'].std()
df['이상']=df['불량률']>mean+3*std
print(f'이상감지: {df["이상"].sum()}건')


## Step 2: 종합 대시보드 생성

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

# 📊 KPI 요약 카드 (상단 3개)
ax_oee = fig.add_subplot(gs[0, 0])
ax_oee.text(0.5, 0.5, f"{df['OEE'].mean():.1f}%", ha='center', va='center',
            fontsize=40, fontweight='bold',
            color='#2ecc71' if df['OEE'].mean()>=FACTORY['target_oee'] else '#e74c3c')
ax_oee.text(0.5,0.15,'평균 OEE',ha='center',fontsize=12,color='gray')
ax_oee.axis('off'); ax_oee.set_title('OEE',fontweight='bold')

ax_def = fig.add_subplot(gs[0, 1])
ax_def.text(0.5,0.5,f"{df['불량률'].mean():.2f}%",ha='center',va='center',
            fontsize=40,fontweight='bold',
            color='#2ecc71' if df['불량률'].mean()<=FACTORY['target_defect'] else '#e74c3c')
ax_def.text(0.5,0.15,'평균 불량률',ha='center',fontsize=12,color='gray')
ax_def.axis('off'); ax_def.set_title('불량률',fontweight='bold')

ax_ano = fig.add_subplot(gs[0, 2])
ax_ano.text(0.5,0.5,f"{df['이상'].sum()}건",ha='center',va='center',
            fontsize=40,fontweight='bold',
            color='#e74c3c' if df['이상'].sum()>0 else '#2ecc71')
ax_ano.text(0.5,0.15,'이상 감지',ha='center',fontsize=12,color='gray')
ax_ano.axis('off'); ax_ano.set_title('예지보전',fontweight='bold')

# 📈 추이 차트 (중간 + 하단)
ax1=fig.add_subplot(gs[1,:])
ax1.plot(df[date_col],df['OEE'],label='OEE',color='#3498db')
ax1.plot(df[date_col],df['불량률']*10,label='불량률×10',color='#e74c3c',linestyle='--')
ax1.axhline(FACTORY['target_oee'],color='orange',linestyle=':',label='OEE목표')
ax1.scatter(df.loc[df['이상'],date_col],df.loc[df['이상'],'OEE'],
            color='red',s=100,zorder=5,label='이상')
ax1.set_title('종합 KPI 추이'); ax1.legend()

ax2=fig.add_subplot(gs[2,:2])
df.set_index(date_col)['가동시간'].plot.area(ax=ax2,color='steelblue',alpha=0.7)
ax2.set_title('가동시간 추이')

ax3=fig.add_subplot(gs[2,2])
metrics=['가동률','불량률 역','OEE']
vals=[df['가동률'].mean(), 100-df['불량률'].mean()*10, df['OEE'].mean()]
ax3.bar(metrics,vals,color=['#3498db','#2ecc71','#9b59b6'])
ax3.set_title('KPI 종합 요약')

plt.suptitle(f'{FACTORY["name"]} AI 종합 운영 대시보드',fontsize=16,fontweight='bold',y=1.01)
plt.savefig('mfg_dashboard.png',dpi=150,bbox_inches='tight')
plt.show()
print('💾 대시보드 저장 완료')


## Step 3: 다운로드

In [ ]:
from google.colab import files
files.download('mfg_dashboard.png')
print('✅ 대시보드 다운로드 완료!')
print('👉 W12: 이 대시보드를 Streamlit 웹앱으로 배포합니다.')


---
## 🔥 확장 과제
1. 색상 테마를 회사 CI 색상으로 변경
2. 월간/주간 필터 추가
3. W12에서 Flask 서버로 배포